In [1]:
pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Fraud Detection End-to-End Pipeline
# - Loads data (card_transdata.csv)
# - EDA summary
# - Train/validation/test split
# - Scaling + SMOTE in Pipelines
# - Trains Logistic Regression, Random Forest, and XGBoost (optional)
# - Evaluates with ROC-AUC, PR-AUC, F1, confusion matrix
# - Plots ROC and Precision-Recall curves
# - Shows feature importance and SHAP (optional if installed)
# Author: Gape Keith Obert

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Imbalanced data handling
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Try optional libraries
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False


# ===== 1) Load data =====
CSV_PATH = "/Users/keith/Downloads/card_transdata.csv"
assert os.path.exists(CSV_PATH), f"File not found: {CSV_PATH}"

df = pd.read_csv(CSV_PATH)

# Basic checks
print("\n=== Head ===")
print(df.head())

print("\n=== Info ===")
print(df.info())

print("\n=== Fraud value counts ===")
print(df["fraud"].value_counts(dropna=False))

# ===== 2) Define features and target =====
# Based on your columns in the screenshot
FEATURES = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order",
]
TARGET = "fraud"

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

# Identify numeric/categorical (all are numeric/binary here, but we scale continuous ones)
numeric_features = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
]
binary_features = [
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order",
]

# ===== 3) Split train/valid/test =====
# Keep a final unseen test set; use validation for model selection
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)  # 0.1765 of 85% ≈ 15%, so final split ~70/15/15

print("\n=== Split sizes ===")
print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")

# ===== 4) Preprocessing: scale numeric, pass-through binary =====
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

# ===== 5) Define models with SMOTE within pipeline =====
# Note: SMOTE should be applied inside the training pipeline (after preprocessing)
# We'll use class weights for LR as an alternative to SMOTE, but leave SMOTE on by default

def make_lr_pipeline(use_smote=True):
    steps = [("prep", preprocess)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    steps.append(("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)))
    return ImbPipeline(steps)

def make_rf_pipeline(use_smote=True):
    steps = [("prep", preprocess)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    steps.append(("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    )))
    return ImbPipeline(steps)

def make_xgb_pipeline(use_smote=True):
    # XGBoost handles imbalance via scale_pos_weight, but we can still try SMOTE
    steps = [("prep", preprocess)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    # Compute scale_pos_weight ~ (negatives/positives) from training data
    pos = int(y_train.sum())
    neg = int((y_train.shape[0] - pos))
    spw = max(1.0, neg / max(1, pos))

    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
        eval_metric="logloss",
        tree_method="hist"
    )
    steps.append(("clf", xgb))
    return ImbPipeline(steps)

models = {
    "LogReg_SMOTE": make_lr_pipeline(use_smote=True),
    "RandForest_SMOTE": make_rf_pipeline(use_smote=True),
}
if XGB_AVAILABLE:
    models["XGBoost_SMOTE"] = make_xgb_pipeline(use_smote=True)

# ===== 6) Train and validate =====
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    # Validation predictions
    yv_pred = pipe.predict(X_valid)
    yv_proba = pipe.predict_proba(X_valid)[:, 1] if hasattr(pipe.named_steps["clf"], "predict_proba") else None

    roc = roc_auc_score(y_valid, yv_proba) if yv_proba is not None else np.nan
    prauc = average_precision_score(y_valid, yv_proba) if yv_proba is not None else np.nan

    print(f"\n=== {name} - Validation Report ===")
    print(classification_report(y_valid, yv_pred, digits=4))
    print("Confusion Matrix (valid):")
    print(confusion_matrix(y_valid, yv_pred))
    print(f"ROC-AUC (valid): {roc:.4f}")
    print(f"PR-AUC  (valid): {prauc:.4f}")

    results.append({
        "model": name,
        "roc_auc_valid": roc,
        "pr_auc_valid": prauc
    })

# Summary of validation performance
summary = pd.DataFrame(results).sort_values(by=["pr_auc_valid", "roc_auc_valid"], ascending=False)
print("\n=== Validation Summary (sorted by PR-AUC then ROC-AUC) ===")
print(summary)

best_name = summary.iloc[0]["model"]
best_model = models[best_name]
print(f"\nSelected best model: {best_name}")

# ===== 7) Retrain on Train+Valid, Evaluate on Test =====
X_trval = pd.concat([X_train, X_valid], axis=0)
y_trval = pd.concat([y_train, y_valid], axis=0)

best_model.fit(X_trval, y_trval)

yt_pred = best_model.predict(X_test)
yt_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model.named_steps["clf"], "predict_proba") else None

print("\n=== Test Report (final) ===")
print(classification_report(y_test, yt_pred, digits=4))
print("Confusion Matrix (test):")
print(confusion_matrix(y_test, yt_pred))

if yt_proba is not None:
    roc_test = roc_auc_score(y_test, yt_proba)
    prauc_test = average_precision_score(y_test, yt_proba)
    print(f"ROC-AUC (test): {roc_test:.4f}")
    print(f"PR-AUC  (test): {prauc_test:.4f}")

# ===== 8) Curves: ROC and Precision-Recall =====
def plot_curves(y_true, y_proba, title_prefix="Test"):
    if y_proba is None:
        print("No probability outputs; skipping curves.")
        return
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"{title_prefix} ROC (AUC={roc_auc_score(y_true, y_proba):.3f})")
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # PR
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, label=f"{title_prefix} PR (AP={average_precision_score(y_true, y_proba):.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} Precision-Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_curves(y_test, yt_proba, title_prefix="Test")

# ===== 9) Feature importance =====
def feature_importance_from_model(fitted_pipe, feature_names):
    clf = fitted_pipe.named_steps["clf"]
    # Get transformed feature names
    # ColumnTransformer reorders: numeric (scaled) + binary passthrough
    transformed_feature_names = numeric_features + binary_features

    if hasattr(clf, "feature_importances_"):
        importances = clf.feature_importances_
        fi = pd.DataFrame({"feature": transformed_feature_names, "importance": importances})
        fi = fi.sort_values("importance", ascending=False)
        print("\n=== Tree-based Feature Importance ===")
        print(fi)
        plt.figure(figsize=(7,4))
        sns.barplot(x="importance", y="feature", data=fi)
        plt.title("Feature Importance")
        plt.tight_layout()
        plt.show()
    elif hasattr(clf, "coef_"):
        coefs = clf.coef_.ravel()
        fi = pd.DataFrame({"feature": transformed_feature_names, "coefficient": coefs})
        fi = fi.sort_values("coefficient", key=np.abs, ascending=False)
        print("\n=== Logistic Regression Coefficients ===")
        print(fi)
        plt.figure(figsize=(7,4))
        sns.barplot(x="coefficient", y="feature", data=fi)
        plt.title("Logistic Coefficients (scaled features)")
        plt.tight_layout()
        plt.show()
    else:
        print("Model does not expose feature importances or coefficients.")

feature_importance_from_model(best_model, FEATURES)

# ===== 10) SHAP explainability (optional) =====
if SHAP_AVAILABLE:
    try:
        print("\n=== SHAP Explainability ===")
        clf = best_model.named_steps["clf"]

        # Fit a SHAP explainer depending on model type
        if hasattr(clf, "predict_proba"):
            # Build a processed sample (apply preprocess only)
            preprocess_only = Pipeline(steps=[("prep", preprocess)])
            X_test_proc = preprocess_only.fit(X_trval, y_trval).transform(X_test)

            if hasattr(clf, "get_booster") or clf.__class__.__name__.startswith("XGB"):
                explainer = shap.TreeExplainer(clf)
                shap_values = explainer.shap_values(X_test_proc)
            elif hasattr(clf, "feature_importances_"):
                explainer = shap.TreeExplainer(clf)
                shap_values = explainer.shap_values(X_test_proc)
            else:
                explainer = shap.LinearExplainer(clf, X_test_proc, feature_dependence="independent")
                shap_values = explainer.shap_values(X_test_proc)

            shap.summary_plot(shap_values, X_test_proc, feature_names=numeric_features + binary_features, show=True)
        else:
            print("SHAP skipped: classifier lacks predict_proba.")
    except Exception as e:
        print(f"SHAP encountered an issue: {e}")

# ===== 11) Save artifacts =====
OUT_DIR = "artifacts_fraud"
os.makedirs(OUT_DIR, exist_ok=True)

# Save validation summary
summary.to_csv(os.path.join(OUT_DIR, "validation_summary.csv"), index=False)

# Save final test predictions
test_out = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": yt_pred,
    "y_proba": yt_proba if yt_proba is not None else np.nan
})
test_out.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

print(f"\nArtifacts saved to: {OUT_DIR}")
print("Done.")



=== Head ===
   distance_from_home  distance_from_last_transaction  \
0           57.877857                        0.311140   
1           10.829943                        0.175592   
2            5.091079                        0.805153   
3            2.247564                        5.600044   
4           44.190936                        0.566486   

   ratio_to_median_purchase_price  repeat_retailer  used_chip  \
0                        1.945940              1.0        1.0   
1                        1.294219              1.0        0.0   
2                        0.427715              1.0        0.0   
3                        0.362663              1.0        1.0   
4                        2.222767              1.0        1.0   

   used_pin_number  online_order  fraud  
0              0.0           0.0    0.0  
1              0.0           0.0    0.0  
2              0.0           1.0    0.0  
3              0.0           1.0    0.0  
4              0.0           1.0    0.0  
